# SoftCAM-Transformer v4 — Run C1 (entraînement from scratch, gating par produit)

**Objectif** : entraîner v4 from scratch sur Cluster 4 et comparer à B5 (v3).

| | v3 (B5) | v4 (C1) |
|--|---------|----------|
| Architecture | h_ev = (1-mix)·dec + mix·LN(bmm) | h_ev = dec ⊙ gate |
| Warm-up mix | ✅ obligatoire (15 epochs mix=0) | ❌ inutile (gate=1 à l'init) |
| Schedule gamma | anneal epochs 25-40 | anneal epochs 25-40 (identique) |
| evidence_mix | 0.05 entraînement / 0.25 inférence | N/A |
| Epochs | 51 | 51 |
| LR | 6e-4 | 6e-4 |

**Pourquoi le warm-up est inutile en v4 :**
`gate_proj.weight ~ N(0, 0.01)` → `gate ≈ 1` à l'epoch 0 → `h_ev ≈ dec_output`.
Le décodeur apprend normalement dès le début. Le gate s'éloigne de 1 progressivement
au fil des epochs, ce qui est exactement ce que le warm-up forçait manuellement en v3.

**GATE H1.C** : R²≥0.30, Spearman≥0.85  
**Cible** : R²≥0.75 (dépasser B5=0.7563)

> Avant de lancer : Runtime → Change runtime type → **T4 GPU**.

## 1 — Setup

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'Pas de GPU — vérifier le runtime.')

In [ ]:
%%capture
!pip install -q transformers datasets "gluonts[torch]" accelerate evaluate scipy scikit-learn tqdm openpyxl ujson

## 2 — Récupération du code

In [ ]:
import os, sys

REPO_URL = 'https://github.com/theblackmamba08/recherche-m2-xai-faas.git'
REPO_DIR = '/content/recherche-m2-xai-faas'

ipy = get_ipython()
if not os.path.isdir(REPO_DIR):
    ipy.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    ipy.system(f'git -C {REPO_DIR} pull')

if f'{REPO_DIR}/code' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/code')

models_dir = f'{REPO_DIR}/code/src/models'
if not os.path.isdir(models_dir):
    raise FileNotFoundError(
        f'{models_dir} introuvable.\n'
        'Essaie : Runtime → Disconnect and delete runtime, puis relance.'
    )
print('Repo prêt :', os.listdir(models_dir))

## 3 — Imports, seed, config

In [ ]:
import random, json, gc
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import lru_cache, partial
from pathlib import Path
from tqdm.notebook import tqdm

SEED = 998
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Hyperparams identiques à B5 ──────────────────────────────────────────────
FREQ              = '1T'
PREDICTION_LENGTH = 120
CONTEXT_LENGTH    = 240
D_MODEL           = 32
N_HEADS           = 2
ENCODER_LAYERS    = 4
DECODER_LAYERS    = 4
EMBEDDING_DIM     = [2]
DROPOUT           = 0.1
BATCH_SIZE_TRAIN  = 256
BATCH_SIZE_TEST   = 64
NUM_BATCHES_EPOCH = 100
LR                = 6e-4
BETAS             = (0.9, 0.95)
WEIGHT_DECAY      = 1e-1
GRAD_CLIP_NORM    = 1.0
NUM_EPOCHS        = 51

# ── Evidence Layer v4 ─────────────────────────────────────────────────────────
USE_EVIDENCE_LAYER   = True
ALPHA_L1             = 0.0
BETA_L2              = 1e-3
GAMMA_ENTROPY_TARGET = 1e-3

# ── Schedule gamma (identique B5 — pas de schedule mix, inutile en v4) ───────
# En v3 : warm-up mix obligatoire (gate additif perturbait dec_output à l'init)
# En v4 : gate_proj ~ N(0,0.01) → gate=1 → h_ev=dec_output à l'init
#          Le warm-up est structurellement intégré dans l'initialisation.
GAMMA_ANNEAL_START = 25
GAMMA_ANNEAL_END   = 40

def gamma_schedule(epoch: int) -> float:
    if epoch < GAMMA_ANNEAL_START:
        return 0.0
    if epoch < GAMMA_ANNEAL_END:
        return GAMMA_ENTROPY_TARGET * (epoch - GAMMA_ANNEAL_START) / (GAMMA_ANNEAL_END - GAMMA_ANNEAL_START)
    return GAMMA_ENTROPY_TARGET

# ── Références ────────────────────────────────────────────────────────────────
FAYAM_R2,  FAYAM_SPEAR  = 0.3701, 0.9201
B5_R2,     B5_SPEAR     = 0.7563, 0.9870
GATE_R2,   GATE_SPEAR   = 0.30,   0.85

CLUSTER_ID = 4
RUN_NAME   = f'softcam-cluster{CLUSTER_ID}-v4-runC1'

print(f'Run         : {RUN_NAME}')
print(f'Modèle      : SoftCAMTransformerV4ForPrediction (gate = 1+tanh)')
print(f'Entraînement: from scratch (SEED={SEED}, {NUM_EPOCHS} epochs, LR={LR})')
print(f'Warm-up mix : ❌ inutile — gate=1 à l\'init par construction')
print(f'Schedule γ  : {{0..{GAMMA_ANNEAL_START-1}}}→0  {{{GAMMA_ANNEAL_START}..{GAMMA_ANNEAL_END-1}}}→ramp  {{{GAMMA_ANNEAL_END}+}}→{GAMMA_ENTROPY_TARGET:.0e}')
print(f'GATE H1.C   : R²≥{GATE_R2}  Spearman≥{GATE_SPEAR}')

## 4 — Visualisation du schedule gamma

In [ ]:
ep_axis    = np.arange(NUM_EPOCHS)
gamma_vals = np.array([gamma_schedule(e) for e in ep_axis])

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(ep_axis, gamma_vals, color='firebrick', lw=2, label='gamma_entropy')
ax.axhline(GAMMA_ENTROPY_TARGET, color='gray', ls=':', alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('gamma_entropy')
ax.set_title(
    'Run C1 — Schedule γ (identique B5)\n'
    'Pas de schedule mix : gate=1 à l\'init → warm-up intégré par construction'
)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5 — Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = f'/content/drive/MyDrive/m2-xai-faas/experiments/{RUN_NAME}'
for subdir in ['checkpoints', 'results', 'logs', 'figures', 'evidence_maps']:
    os.makedirs(f'{DRIVE_BASE}/{subdir}', exist_ok=True)
print(f'Dossier run : {DRIVE_BASE}')

## 6 — Chargement Cluster 4

In [ ]:
from datasets import Dataset

DATA_DIR   = Path('/content/drive/MyDrive/Recherche/Datasets')
START_DATE = '2021-01-01 00:00:00'

df = pd.read_csv(DATA_DIR / f'cluster_{CLUSTER_ID}.csv', index_col='Function')
all_series = []
for func_id, row in df.iterrows():
    all_series.append({
        'function_id': int(func_id),
        'cluster':     CLUSTER_ID,
        'target_full': row.values.astype(np.float32),
    })

print(f'Cluster {CLUSTER_ID} — {len(all_series)} fonctions')
for s in all_series:
    print(f"  fn={s['function_id']}  len={len(s['target_full'])}  "
          f"moy={s['target_full'].mean():.0f}  max={s['target_full'].max():.0f}")

train_rows, test_rows = [], []
for ts_idx, s in enumerate(all_series):
    base = {'start': START_DATE, 'feat_static_cat': [ts_idx],
            'cluster': s['cluster'], 'function_id': s['function_id']}
    train_rows.append({**base, 'target': s['target_full'][:-PREDICTION_LENGTH].tolist()})
    test_rows.append( {**base, 'target': s['target_full'].tolist()})

train_dataset = Dataset.from_list(train_rows)
test_dataset  = Dataset.from_list(test_rows)
print(f'\nDatasets : train={len(train_dataset)}  test={len(test_dataset)}')

## 7 — Pipeline GluonTS (identique B5)

In [ ]:
from gluonts.time_feature import get_lags_for_frequency, time_features_from_frequency_str
from gluonts.dataset.field_names import FieldName
from gluonts.transform import (
    AddAgeFeature, AddObservedValuesIndicator, AddTimeFeatures,
    AsNumpyArray, Chain, ExpectedNumInstanceSampler, InstanceSplitter,
    RemoveFields, TestSplitSampler, ValidationSplitSampler,
    VstackFeatures, RenameFields,
)
from gluonts.itertools import Cyclic, Cached
from gluonts.dataset.loader import as_stacked_batches

lags_sequence = get_lags_for_frequency(FREQ)
time_features  = time_features_from_frequency_str(FREQ)

@lru_cache(10_000)
def convert_to_pandas_period(date, freq):
    return pd.Period(date, freq)

def transform_start_field(batch, freq):
    batch['start'] = [convert_to_pandas_period(d, freq) for d in batch['start']]
    return batch

for ds in (train_dataset, test_dataset):
    ds.set_transform(partial(transform_start_field, freq=FREQ))

def create_transformation(freq, config):
    remove_field_names = []
    if config.num_static_real_features == 0:
        remove_field_names.append(FieldName.FEAT_STATIC_REAL)
    if config.num_dynamic_real_features == 0:
        remove_field_names.append(FieldName.FEAT_DYNAMIC_REAL)
    if config.num_static_categorical_features == 0:
        remove_field_names.append(FieldName.FEAT_STATIC_CAT)
    return Chain(
        [RemoveFields(field_names=remove_field_names)]
        + ([AsNumpyArray(field=FieldName.FEAT_STATIC_CAT, expected_ndim=1, dtype=int)]
           if config.num_static_categorical_features > 0 else [])
        + [AsNumpyArray(field=FieldName.TARGET, expected_ndim=1),
           AddObservedValuesIndicator(target_field=FieldName.TARGET,
                                      output_field=FieldName.OBSERVED_VALUES),
           AddTimeFeatures(start_field=FieldName.START, target_field=FieldName.TARGET,
                           output_field=FieldName.FEAT_TIME,
                           time_features=time_features_from_frequency_str(freq),
                           pred_length=config.prediction_length),
           AddAgeFeature(target_field=FieldName.TARGET, output_field=FieldName.FEAT_AGE,
                         pred_length=config.prediction_length, log_scale=True),
           VstackFeatures(output_field=FieldName.FEAT_TIME,
                          input_fields=[FieldName.FEAT_TIME, FieldName.FEAT_AGE]),
           RenameFields(mapping={
               FieldName.FEAT_STATIC_CAT: 'static_categorical_features',
               FieldName.FEAT_TIME:       'time_features',
               FieldName.TARGET:          'values',
               FieldName.OBSERVED_VALUES: 'observed_mask',
           })]
    )

def create_instance_splitter(config, mode):
    sampler = {
        'train':      ExpectedNumInstanceSampler(num_instances=1.0,
                                                  min_future=config.prediction_length),
        'validation': ValidationSplitSampler(min_future=config.prediction_length),
        'test':       TestSplitSampler(),
    }[mode]
    return InstanceSplitter(
        target_field='values', is_pad_field=FieldName.IS_PAD,
        start_field=FieldName.START, forecast_start_field=FieldName.FORECAST_START,
        instance_sampler=sampler,
        past_length=config.context_length + max(config.lags_sequence),
        future_length=config.prediction_length,
        time_series_fields=['time_features', 'observed_mask'],
    )

def _pred_fields(config):
    f = ['past_time_features', 'past_values', 'past_observed_mask', 'future_time_features']
    if config.num_static_categorical_features > 0:
        f.append('static_categorical_features')
    return f

def create_train_dataloader(config, freq, data, batch_size, num_batches_per_epoch):
    tr = create_transformation(freq, config)
    sp = create_instance_splitter(config, 'train')
    # Cyclic(Cached(...)) : cache le dataset fini une fois, puis cycle.
    # NE PAS inverser en Cached(Cyclic(...)) — Cyclic est infini → OOM immédiat.
    return as_stacked_batches(
        Cyclic(Cached(sp.apply(tr.apply(data, is_train=True), is_train=True))),
        batch_size=batch_size,
        num_batches_per_epoch=num_batches_per_epoch,
        output_type=torch.tensor,
        field_names=['past_time_features', 'past_values', 'past_observed_mask',
                     'future_time_features', 'future_values', 'future_observed_mask']
        + (['static_categorical_features'] if config.num_static_categorical_features > 0 else []),
    )

def create_backtest_dataloader(config, freq, data, batch_size):
    tr = create_transformation(freq, config)
    sp = create_instance_splitter(config, 'validation')
    return as_stacked_batches(sp.apply(tr.apply(data), is_train=True),
                              batch_size=batch_size, output_type=torch.tensor,
                              field_names=_pred_fields(config))

print('Pipeline GluonTS prête.')

## 8 — Construction du modèle v4 (from scratch)

In [ ]:
from src.models.softcam_transformer_v4 import (
    SoftCAMTransformerV4Config,
    SoftCAMTransformerV4ForPrediction,
)

cfg = SoftCAMTransformerV4Config(
    prediction_length=PREDICTION_LENGTH,
    context_length=CONTEXT_LENGTH,
    lags_sequence=lags_sequence,
    num_time_features=len(time_features) + 1,
    num_static_categorical_features=1,
    cardinality=[len(train_dataset)],
    embedding_dimension=EMBEDDING_DIM,
    encoder_layers=ENCODER_LAYERS,
    decoder_layers=DECODER_LAYERS,
    d_model=D_MODEL,
    encoder_attention_heads=N_HEADS,
    decoder_attention_heads=N_HEADS,
    encoder_ffn_dim=max(D_MODEL, 32),
    decoder_ffn_dim=max(D_MODEL, 32),
    dropout=DROPOUT,
    use_evidence_layer=USE_EVIDENCE_LAYER,
    evidence_mix=0.05,   # conservé pour compat config, ignoré par v4
    alpha_l1=ALPHA_L1,
    beta_l2=BETA_L2,
    gamma_entropy=GAMMA_ENTROPY_TARGET,
)

model = SoftCAMTransformerV4ForPrediction(cfg).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Modèle v4 initialisé from scratch.')
print(f'Paramètres : {total_params:,}')
print()

# Vérification gate ≈ identité au démarrage
gw = model.ev_layer_v4.gate_proj.weight
gb = model.ev_layer_v4.gate_proj.bias
print(f'gate_proj.weight : std={gw.std().item():.4f}  (attendu ≈ 0.01)')
print(f'gate_proj.bias   : mean={gb.mean().item():.4f}  (attendu = 0)')
print(f'→ gate ≈ 1+tanh(0) = 1 → h_ev ≈ dec_output dès l\'epoch 0 ✅')

## 9 — Entraînement from scratch (51 epochs)

In [ ]:
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=BETAS, weight_decay=WEIGHT_DECAY
)

train_loader = create_train_dataloader(
    cfg, FREQ, train_dataset, BATCH_SIZE_TRAIN, NUM_BATCHES_EPOCH
)

history = []

for epoch in range(NUM_EPOCHS):
    gamma = gamma_schedule(epoch)
    model.gamma_entropy = gamma

    model.train()
    epoch_losses = {'total': [], 'forecast': [], 'elastic': [], 'entropy': []}

    for batch in train_loader:
        optimizer.zero_grad()
        out = model(
            static_categorical_features=batch['static_categorical_features'].to(device)
                if cfg.num_static_categorical_features > 0 else None,
            past_time_features=batch['past_time_features'].to(device),
            past_values=batch['past_values'].to(device),
            past_observed_mask=batch['past_observed_mask'].to(device),
            future_time_features=batch['future_time_features'].to(device),
            future_values=batch['future_values'].to(device),
            future_observed_mask=batch['future_observed_mask'].to(device),
        )
        if out.loss is None:
            continue
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        epoch_losses['total'].append(out.loss.item())
        if out.forecast_loss is not None:
            epoch_losses['forecast'].append(out.forecast_loss.item())
        if out.elastic_loss is not None:
            epoch_losses['elastic'].append(out.elastic_loss.item())
        if out.entropy_loss is not None:
            epoch_losses['entropy'].append(out.entropy_loss.item())

    mean_loss = np.mean(epoch_losses['total'])
    mean_fc   = np.mean(epoch_losses['forecast']) if epoch_losses['forecast'] else float('nan')
    mean_ent  = np.mean(epoch_losses['entropy'])  if epoch_losses['entropy']  else float('nan')

    history.append({'epoch': epoch, 'loss': mean_loss, 'forecast': mean_fc,
                    'entropy': mean_ent, 'gamma': gamma})

    print(f'Epoch {epoch:3d}/{NUM_EPOCHS-1}  '
          f'loss={mean_loss:.4f}  fc={mean_fc:.4f}  '
          f'H={mean_ent:.4f}  γ={gamma:.1e}')

print('\nEntraînement terminé.')

## 10 — Courbes de loss

In [ ]:
epochs_ax = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_ax, [h['loss']     for h in history], color='steelblue', lw=2)
axes[0].set_title('Loss totale'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ax, [h['forecast'] for h in history], color='firebrick', lw=2)
axes[1].set_title('Loss forecast (NLL Student-T)'); axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_ax, [h['entropy']  for h in history], color='seagreen', lw=2)
axes[2].set_title('Terme entropie γ·H(M)'); axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3)

plt.suptitle(f'{RUN_NAME} — From scratch ({NUM_EPOCHS} epochs, LR={LR})', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/figures/loss_curve.png', dpi=150)
plt.show()

## 11 — Sauvegarde checkpoint v4

In [ ]:
ckpt_v4_path = f'{DRIVE_BASE}/checkpoints/v4_runC1_final.pt'
torch.save({
    'model':      model.state_dict(),
    'optimizer':  optimizer.state_dict(),
    'config':     cfg.to_dict(),
    'history':    history,
    'run_name':   RUN_NAME,
    'num_epochs': NUM_EPOCHS,
    'lr':         LR,
    'seed':       SEED,
}, ckpt_v4_path)
print(f'Checkpoint sauvegardé : {ckpt_v4_path}')
print(f'Taille : {os.path.getsize(ckpt_v4_path) / 1e6:.1f} MB')

## 12 — Évaluation finale (generate + médiane 100 trajectoires)

In [ ]:
@torch.no_grad()
def evaluate_generate(model, dataloader, dataset, config, device):
    model.eval()
    forecasts = []
    for b in dataloader:
        out = model.generate(
            static_categorical_features=b['static_categorical_features'].to(device)
                if config.num_static_categorical_features > 0 else None,
            past_time_features=b['past_time_features'].to(device),
            past_values=b['past_values'].to(device),
            future_time_features=b['future_time_features'].to(device),
            past_observed_mask=b['past_observed_mask'].to(device),
        )
        forecasts.append(out.sequences.cpu().numpy())
    forecasts = np.vstack(forecasts)
    forecast_median = np.median(forecasts, axis=1)

    r2s, spears = [], []
    for ts_idx in range(len(dataset)):
        target = np.array(dataset[ts_idx]['target'])
        actual = target[-config.prediction_length:]
        pred   = forecast_median[ts_idx]
        r2s.append(r2_score(actual, pred))
        rho, _ = spearmanr(actual, pred)
        spears.append(rho)
    return float(np.mean(r2s)), float(np.mean(spears)), r2s, spears

test_loader = create_backtest_dataloader(cfg, FREQ, test_dataset, BATCH_SIZE_TEST)
r2, spear, r2s, spears = evaluate_generate(model, test_loader, test_dataset, cfg, device)

gate_h1c = r2 >= GATE_R2 and spear >= GATE_SPEAR
vs_b5    = r2 - B5_R2
vs_fayam = r2 - FAYAM_R2

print('=' * 60)
print(f'  {RUN_NAME} — Résultats finals')
print('=' * 60)
print(f'  R²       = {r2:.4f}   (FAYAM={FAYAM_R2:.4f}  B5={B5_R2:.4f})')
print(f'  Spearman = {spear:.4f}   (FAYAM={FAYAM_SPEAR:.4f}  B5={B5_SPEAR:.4f})')
print()
print(f'  Per-series R²       : {[f"{x:.3f}" for x in r2s]}')
print(f'  Per-series Spearman : {[f"{x:.3f}" for x in spears]}')
print()
print(f'  GATE H1.C : {"PASS ✅" if gate_h1c else "FAIL ❌"}')
print(f'  vs B5     : {vs_b5:+.4f} pp')
print(f'  vs FAYAM  : {vs_fayam:+.4f} pp')
print('=' * 60)

## 13 — gate_deviation : M est-elle utilisée effectivement ?

In [ ]:
test_loader_single = create_backtest_dataloader(cfg, FREQ, test_dataset, batch_size=1)
deviations = []

for ts_idx, b in enumerate(test_loader_single):
    func_id = test_dataset[ts_idx]['function_id']
    target_full = np.array(test_dataset[ts_idx]['target'])
    future_vals = torch.tensor(
        target_full[-PREDICTION_LENGTH:], dtype=torch.float32
    ).unsqueeze(0).to(device)
    future_obs = torch.ones_like(future_vals)

    dev = model.gate_deviation(
        past_values=b['past_values'].to(device),
        past_time_features=b['past_time_features'].to(device),
        past_observed_mask=b['past_observed_mask'].to(device),
        future_values=future_vals,
        future_time_features=b['future_time_features'].to(device),
        static_categorical_features=b['static_categorical_features'].to(device)
            if cfg.num_static_categorical_features > 0 else None,
        future_observed_mask=future_obs,
    )
    deviations.append((func_id, dev))
    print(f'  fn {func_id}  gate_deviation = {dev:.4f}')

mean_dev = np.mean([d for _, d in deviations])
print(f'\ngate_deviation moyen = {mean_dev:.4f}')
print(f'→ {"M utilisée ✅" if mean_dev > 0.05 else "M peu utilisée ⚠️"}')

## 14 — Extraction des cartes M

In [ ]:
evidence_maps_v4 = []
model.eval()
with torch.no_grad():
    for ts_idx, b in enumerate(test_loader_single):
        func_id = test_dataset[ts_idx]['function_id']
        target_full = np.array(test_dataset[ts_idx]['target'])
        future_vals = torch.tensor(
            target_full[-PREDICTION_LENGTH:], dtype=torch.float32
        ).unsqueeze(0).to(device)
        future_obs = torch.ones_like(future_vals)

        M = model.explain(
            past_values=b['past_values'].to(device),
            past_time_features=b['past_time_features'].to(device),
            past_observed_mask=b['past_observed_mask'].to(device),
            future_values=future_vals,
            future_time_features=b['future_time_features'].to(device),
            static_categorical_features=b['static_categorical_features'].to(device)
                if cfg.num_static_categorical_features > 0 else None,
            future_observed_mask=future_obs,
        )
        M_np = M.squeeze(0).cpu().numpy()
        evidence_maps_v4.append((func_id, M_np))
        np.save(f'{DRIVE_BASE}/evidence_maps/M_func{func_id}.npy', M_np)
        print(f'  fn {func_id}  argmax_mean={int(M_np.argmax(axis=1).mean())}  '
              f'max_weight={M_np.max():.4f}  '
              f'entropy={-(M_np * np.log(M_np.clip(1e-9))).sum(axis=1).mean():.3f}')

fig, axes = plt.subplots(1, len(evidence_maps_v4), figsize=(4*len(evidence_maps_v4), 4))
if len(evidence_maps_v4) == 1:
    axes = [axes]
M_max = max(m.max() for _, m in evidence_maps_v4)
for ax, (fid, M_np) in zip(axes, evidence_maps_v4):
    im = ax.imshow(M_np, aspect='auto', cmap='YlOrRd', vmin=0, vmax=M_max, origin='upper')
    ax.set_title(f'M v4 [fn {fid}]')
    ax.set_xlabel('Contexte encodeur'); ax.set_ylabel('Horizon t')
    plt.colorbar(im, ax=ax, fraction=0.04)
plt.suptitle(f'{RUN_NAME} — Cartes M', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/figures/evidence_maps.png', dpi=150)
plt.show()

## 15 — Prédictions par série

In [ ]:
test_loader_eval = create_backtest_dataloader(cfg, FREQ, test_dataset, BATCH_SIZE_TEST)
forecasts_all = []
model.eval()
with torch.no_grad():
    for b in test_loader_eval:
        out = model.generate(
            static_categorical_features=b['static_categorical_features'].to(device)
                if cfg.num_static_categorical_features > 0 else None,
            past_time_features=b['past_time_features'].to(device),
            past_values=b['past_values'].to(device),
            future_time_features=b['future_time_features'].to(device),
            past_observed_mask=b['past_observed_mask'].to(device),
        )
        forecasts_all.append(out.sequences.cpu().numpy())
forecasts_all   = np.vstack(forecasts_all)
forecast_median = np.median(forecasts_all, axis=1)
forecast_q10    = np.quantile(forecasts_all, 0.1, axis=1)
forecast_q90    = np.quantile(forecasts_all, 0.9, axis=1)

fig, axes = plt.subplots(1, len(test_dataset), figsize=(5*len(test_dataset), 4))
if len(test_dataset) == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    func_id = test_dataset[i]['function_id']
    actual  = np.array(test_dataset[i]['target'])[-PREDICTION_LENGTH:]
    pred    = forecast_median[i]
    t       = np.arange(PREDICTION_LENGTH)
    ax.plot(t, actual, color='black', lw=1.5, label='Réel')
    ax.plot(t, pred,   color='#D97706', lw=1.5, label='v4 médiane')
    ax.fill_between(t, forecast_q10[i], forecast_q90[i], alpha=0.2, color='#D97706')
    ax.set_title(f'fn {func_id}  R²={r2s[i]:.3f}  ρ={spears[i]:.3f}')
    ax.set_xlabel('Horizon (min)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle(f'{RUN_NAME} — Prédictions (médiane + IC 80%)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/figures/forecast_samples.png', dpi=150)
plt.show()

## 16 — Sauvegarde résultats JSON + synthèse finale

In [ ]:
func_ids = [test_dataset[i]['function_id'] for i in range(len(test_dataset))]

results = {
    'run':           RUN_NAME,
    'architecture':  'SoftCAMTransformerV4ForPrediction (gating par produit, from scratch)',
    'num_epochs':    NUM_EPOCHS,
    'lr':            LR,
    'seed':          SEED,
    'r2_mean':       r2,
    'spearman_mean': spear,
    'r2_per_series':       {int(fid): float(v) for fid, v in zip(func_ids, r2s)},
    'spearman_per_series': {int(fid): float(v) for fid, v in zip(func_ids, spears)},
    'gate_deviation_per_series': {int(fid): float(dev) for fid, dev in deviations},
    'gate_deviation_mean': float(mean_dev),
    'vs_fayam_pp':   float(vs_fayam * 100),
    'vs_b5_pp':      float(vs_b5 * 100),
    'gate_h1c':      bool(gate_h1c),
    'references': {
        'fayam': {'r2': FAYAM_R2, 'spearman': FAYAM_SPEAR},
        'b5_v3': {'r2': B5_R2,    'spearman': B5_SPEAR},
    },
}

with open(f'{DRIVE_BASE}/results/test_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Résultats sauvegardés.')

print()
print('=' * 68)
print(f'  SYNTHÈSE — {RUN_NAME}')
print('=' * 68)
print(f'  Architecture  : v4 (gate = 1+tanh, h_ev = dec ⊙ gate)')
print(f'  Entraînement  : from scratch, {NUM_EPOCHS} epochs, LR={LR}, SEED={SEED}')
print(f'  Warm-up mix   : ❌ inutile (gate=1 à l\'init par construction)')
print()
print(f'  R²       = {r2:.4f}   Δ vs B5 = {vs_b5:+.4f}   Δ vs FAYAM = {vs_fayam:+.4f}')
print(f'  Spearman = {spear:.4f}')
print(f'  gate_dev = {mean_dev:.4f}  ({"M utilisée ✅" if mean_dev > 0.05 else "M peu utilisée ⚠️"})')
print()
print(f'  GATE H1.C : {"PASS ✅" if gate_h1c else "FAIL ❌"}')
if vs_b5 > 0:
    print(f'  v4 > v3 : +{vs_b5:.4f} ✅  → v4 devient la configuration de référence')
elif vs_b5 > -0.05:
    print(f'  v4 ≈ v3 : {vs_b5:.4f}  → dans la marge de bruit, architecture équivalente')
else:
    print(f'  v4 < v3 : {vs_b5:.4f}  → inspecter courbes de loss, envisager plus d\'epochs')
print('=' * 68)